# Bài 4.5 · Zero-shot với CLIP và GPT-2

Zero-shot là dùng model **không học thêm** trên dữ liệu của bài. Notebook làm ba thí nghiệm trên dữ liệu thật:

1. **CLIP phân loại CIFAR-10** chỉ từ tên lớp. So ba cách viết prompt, rồi so với linear probe học từ 1, 4, 16 ảnh mỗi lớp.
2. **GPT-2 chọn đoạn kết** cho câu chuyện (HellaSwag): chấm mỗi phương án bằng tổng log xác suất hay trung bình log xác suất theo token.
3. **Perplexity** của GPT-2 trên WikiText-2 đổi theo cách cắt văn bản, nên chỉ so được khi cùng giao thức.

Bật **GPU** và **Internet**. Chạy hết khoảng 8 phút.

In [ ]:
import time, math
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torchvision
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression
from transformers import CLIPModel, CLIPProcessor, AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

torch.manual_seed(0)
import transformers
transformers.logging.set_verbosity_error()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('thiết bị:', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')
N_TEST, N_HELLA, N_TOKENS = 2000, 500, 30000

## 1. CLIP: từ tên lớp đến bộ phân loại

CLIP có hai encoder, một cho ảnh, một cho chữ, học sao cho ảnh và chú thích đúng của nó có embedding gần nhau. Muốn phân loại: mã hóa mỗi tên lớp thành một vector chữ, mã hóa ảnh, rồi chọn lớp có cosine lớn nhất. Không có bước train nào.

In [ ]:
clip = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device).eval()
proc = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
from datasets import load_dataset
cifar = load_dataset('uoft-cs/cifar10')

class HFImages:
    # Bộ ảnh Hugging Face dùng như dataset torchvision: ds[i] trả (ảnh PIL, nhãn).
    def __init__(self, split):
        self.d, self.targets, self.classes = split, list(split['label']), split.features['label'].names
    def __len__(self):
        return len(self.d)
    def __getitem__(self, i):
        r = self.d[int(i)]
        return r['img'].convert('RGB'), r['label']
test, train = HFImages(cifar['test']), HFImages(cifar['train'])
classes = test.classes
rng = np.random.default_rng(0)
te_idx = rng.permutation(len(test))[:N_TEST]
y_te = np.array(test.targets)[te_idx]

# transformers 4 trả thẳng tensor; transformers 5 trả một object, vector chiếu nằm ở pooler_output.
feat = lambda out: out if torch.is_tensor(out) else out.pooler_output

def image_emb(ds, idx, batch=256):
    out = []
    with torch.inference_mode():
        for s in range(0, len(idx), batch):
            imgs = [ds[i][0] for i in idx[s:s + batch]]
            x = proc(images=imgs, return_tensors='pt')['pixel_values'].to(device)
            out.append(torch.nn.functional.normalize(feat(clip.get_image_features(pixel_values=x)), dim=1).cpu())
    return torch.cat(out)

def text_emb(prompts):
    with torch.inference_mode():
        t = proc(text=prompts, return_tensors='pt', padding=True).to(device)
        return torch.nn.functional.normalize(feat(clip.get_text_features(**t)), dim=1).cpu()

t = time.time(); img_te = image_emb(test, te_idx)
print(img_te.shape, f'{time.time() - t:.0f} giây')

In [ ]:
templates = ['a photo of a {}.', 'a blurry photo of a {}.', 'a low resolution photo of a {}.',
             'a close-up photo of a {}.', 'a photo of the small {}.', 'a pixelated photo of a {}.']
ways = {
    'chỉ tên lớp': text_emb(classes),
    '"a photo of a …"': text_emb([f'a photo of a {c}.' for c in classes]),
    'trung bình 6 prompt': torch.nn.functional.normalize(
        torch.stack([text_emb([tp.format(c) for c in classes]) for tp in templates]).mean(0), dim=1),
}
rows = []
for name, txt in ways.items():
    pred = (img_te @ txt.T).argmax(1).numpy()
    rows.append((name, (pred == y_te).mean()))
zs = pd.DataFrame(rows, columns=['cách viết prompt', 'accuracy zero-shot']).round(3)
zs

## 2. Zero-shot so với few-shot

Có vài ảnh có nhãn thì học được một linear probe trên chính embedding ảnh của CLIP. Câu hỏi thực tế: cần bao nhiêu ảnh mỗi lớp thì vượt zero-shot? Mỗi mức lặp 5 lần với các ảnh khác nhau, lấy trung bình.

In [ ]:
y_tr_all = np.array(train.targets)
pool = rng.permutation(len(train))[:3000]
img_pool = image_emb(train, pool).numpy()
y_pool = y_tr_all[pool]
few = []
for k in [1, 4, 16]:
    accs = []
    for rep in range(5):
        r = np.random.default_rng(rep)
        pick = np.concatenate([r.choice(np.flatnonzero(y_pool == c), k, replace=False) for c in range(10)])
        lr = LogisticRegression(max_iter=3000, C=1.0).fit(img_pool[pick], y_pool[pick])
        accs.append((lr.predict(img_te.numpy()) == y_te).mean())
    few.append((k, np.mean(accs), np.std(accs)))
few = pd.DataFrame(few, columns=['ảnh mỗi lớp', 'acc trung bình', 'độ lệch chuẩn'])
best_zs = zs['accuracy zero-shot'].max()
fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(few['ảnh mỗi lớp'], few['acc trung bình'], few['độ lệch chuẩn'], marker='o', capsize=4, label='linear probe')
ax.axhline(best_zs, color='C3', ls='--', label='zero-shot tốt nhất')
ax.set_xscale('log'); ax.set_xticks([1, 4, 16]); ax.set_xticklabels([1, 4, 16])
ax.set_xlabel('số ảnh có nhãn mỗi lớp'); ax.set_ylabel('accuracy test'); ax.legend(); ax.grid(alpha=.3)
plt.show()
few.round(3)

In [ ]:
from sklearn.metrics import confusion_matrix
pred = (img_te @ ways['trung bình 6 prompt'].T).argmax(1).numpy()
cm = pd.DataFrame(confusion_matrix(y_te, pred), index=classes, columns=classes)
off = cm.where(~np.eye(10, dtype=bool)).stack().sort_values(ascending=False).head(6)
print('các cặp nhầm nhiều nhất (thật → đoán):')
off

## 3. GPT-2 chọn đoạn kết

Mỗi câu HellaSwag có một đoạn ngữ cảnh và bốn đoạn kết; chỉ một đoạn hợp lý. Một model ngôn ngữ chấm mỗi đoạn kết bằng xác suất nó sinh ra đoạn ấy sau ngữ cảnh:

$$\log P(\text{kết} \mid \text{ngữ cảnh}) = \sum_t \log P(x_t \mid x_{<t}).$$

Tổng log thiên vị đoạn kết **ngắn**, vì mỗi token thêm vào cộng một số âm. Chia cho số token (trung bình log) bỏ được thiên vị ấy. Ngẫu nhiên đoán được 25%.

In [ ]:
gtok = AutoTokenizer.from_pretrained('gpt2')
gpt = AutoModelForCausalLM.from_pretrained('gpt2').to(device).eval()
hella = load_dataset('Rowan/hellaswag', split='validation').shuffle(seed=0).select(range(N_HELLA))

def ending_logprobs(ctx, ending):
    c = gtok(ctx, return_tensors='pt')['input_ids'][0]
    e = gtok(' ' + ending, return_tensors='pt')['input_ids'][0]
    ids = torch.cat([c, e])[None].to(device)
    with torch.inference_mode():
        logp = gpt(ids).logits.log_softmax(-1)[0, :-1]
    tgt = ids[0, 1:]
    lp = logp[torch.arange(len(tgt)), tgt][len(c) - 1:]
    return lp.sum().item(), lp.mean().item(), len(e)

t = time.time(); hit_sum = hit_mean = 0; lengths = []
for ex in hella:
    scores = [ending_logprobs(ex['ctx'], end) for end in ex['endings']]
    label = int(ex['label'])
    hit_sum += int(np.argmax([s[0] for s in scores]) == label)
    hit_mean += int(np.argmax([s[1] for s in scores]) == label)
    lengths.append((scores[np.argmax([s[0] for s in scores])][2], np.mean([s[2] for s in scores])))
print(f'tổng log: {hit_sum / N_HELLA:.3f} | trung bình log: {hit_mean / N_HELLA:.3f} | ngẫu nhiên: 0.250 | {time.time() - t:.0f} giây')
L = np.array(lengths)
print(f'độ dài (token) của đoạn kết được chọn theo tổng log: {L[:, 0].mean():.1f}, trung bình mọi đoạn kết: {L[:, 1].mean():.1f}')

## 4. Perplexity phụ thuộc giao thức

Perplexity là $\exp$ của trung bình $-\log P$ trên các token. Cùng model, cùng văn bản, nhưng nếu cắt văn bản thành các đoạn ngắn, mỗi token chỉ thấy ít ngữ cảnh phía trước, nên đoán khó hơn và perplexity cao hơn. Vì vậy hai con số perplexity chỉ so được khi cùng tokenizer, cùng văn bản và cùng cách cắt.

In [ ]:
wiki = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='test')
ids = gtok('\n\n'.join(list(wiki['text'])), return_tensors='pt')['input_ids'][0][:N_TOKENS]

def perplexity(window, stride=None):
    stride = stride or window
    nll, count, prev_end = 0.0, 0, 0
    for begin in range(0, len(ids) - 1, stride):
        end = min(begin + window, len(ids))
        x = ids[begin:end][None].to(device)
        with torch.inference_mode():
            logp = gpt(x).logits.log_softmax(-1)[0, :-1]
        tgt = x[0, 1:]
        lp = logp[torch.arange(len(tgt)), tgt]
        new = end - max(prev_end, begin + 1)      # chỉ tính token chưa được chấm ở cửa sổ trước
        nll -= lp[-new:].sum().item(); count += new; prev_end = end
        if end == len(ids):
            break
    return math.exp(nll / count)

rows = []
for window, stride in [(32, None), (128, None), (512, None), (1024, None), (1024, 512)]:
    t = time.time()
    rows.append((window, stride or window, perplexity(window, stride), round(time.time() - t)))
pd.DataFrame(rows, columns=['cửa sổ (token)', 'bước trượt', 'perplexity', 'giây']).round(2)

## Đọc kết quả

Số dưới đây là của lần chạy mẫu trên GPU T4; chạy lại có thể lệch chút ít.

- **CLIP zero-shot** trên 2000 ảnh CIFAR-10: chỉ tên lớp 0,866; câu “a photo of a …” 0,882; trung bình 6 prompt 0,889. Viết tên lớp thành một câu giống chú thích ảnh, và lấy trung bình nhiều cách viết, đều giúp, dù không học thêm gì.
- **Few-shot**: linear probe trên chính embedding CLIP đạt 0,634 với 1 ảnh mỗi lớp, 0,828 với 4 ảnh, 0,879 với 16 ảnh, vẫn chưa bằng zero-shot tốt nhất. Một ảnh mỗi lớp còn dao động mạnh giữa các lần chọn (độ lệch chuẩn 0,059): chọn trúng ảnh xấu là kết quả tệ hẳn.
- **Nhầm nhiều nhất**: hươu thành ngựa (24 ảnh), chó thành mèo (19), mèo thành chó (16), ếch thành chim (14). Ảnh 32 × 32 bị phóng lên 224 nên các loài có dáng gần nhau khó tách.
- **GPT-2 chọn đoạn kết** trên 500 câu HellaSwag: tổng log đúng 29,4%, trung bình log 30,4%, chỉ hơn đoán ngẫu nhiên 25% một chút. Theo tổng log, đoạn kết được chọn dài trung bình 25,2 token, ngắn hơn mức trung bình 31,1 token của mọi đoạn kết: đúng thiên vị đoạn ngắn đã nói.
- **Perplexity** của cùng GPT-2 trên cùng văn bản: 154,33 khi cắt thành đoạn 32 token, 65,76 với 128, 37,81 với 512, 31,22 với 1024, và 26,22 khi cửa sổ 1024 trượt mỗi lần 512 token. Hai con số perplexity chỉ so được khi cùng cách cắt.

Gợi ý khi demo: thêm vài prompt tiếng Việt vào `templates` rồi so accuracy, hoặc đổi `gpt2` thành `gpt2-medium` rồi so accuracy HellaSwag.